In [3]:
import numpy as np
import logging
from typing import Protocol, Dict, Tuple
from pydantic import BaseModel, Field, ValidationError
from datetime import datetime, timezone
import threading


logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


class InvalidBeliefError(Exception):
    """Raised when belief state is invalid or inconsistent."""
    pass


class UpdateError(Exception):
    """Raised when belief update fails."""
    pass


class BeliefStoreError(Exception):
    """Raised when store operation fails."""
    pass


class VsIndex(BaseModel):
    lat: float = Field(..., ge=-90.0, le=90.0)
    lon: float = Field(..., ge=-180.0, le=180.0)
    depth: float = Field(..., ge=-10.0, le=0.0)
    soilsat: float = Field(..., ge=0.0, le=1.0)


class VsMeasurement(BaseModel):
    index: VsIndex
    vs: float = Field(..., ge=0.0, le=5.0)
    sigma: float = Field(..., gt=0.0)
    timestamp: datetime


class BeliefRepresentation(Protocol):
    """Interface for belief representations over uncertain quantities."""
    
    def mean(self) -> float:
        ...
    
    def variance(self) -> float:
        ...
    
    def distribution_family(self) -> str:
        ...


class GaussianBelief:
    
    def __init__(self, mean: float, variance: float) -> None:
        if variance <= 0:
            logger.error(f"Invalid variance: {variance}")
            raise InvalidBeliefError(f"Variance must be positive, got {variance}")
        
        if variance < 0.01:
            logger.warning(f"Variance is very small: {variance}")
        
        self._mean = mean
        self._variance = variance
        logger.info(f"GaussianBelief initialized with mean={mean}, variance={variance}")
    
    def mean(self) -> float:
        return self._mean
    
    def variance(self) -> float:
        return self._variance
    
    def distribution_family(self) -> str:
        return "gaussian"


class UpdateMechanism(Protocol):
    """Interface for updating beliefs given new evidence."""
    
    def apply(
        self, prior: BeliefRepresentation, evidence: VsMeasurement
    ) -> BeliefRepresentation:
        ...


class PrecisionWeightedGaussianUpdate:
    
    def apply(
        self, prior: GaussianBelief, evidence: VsMeasurement
    ) -> GaussianBelief:
        try:
            logger.info(f"Applying update at index {evidence.index}")
            logger.info(f"Prior: mean={prior.mean()}, variance={prior.variance()}")
            logger.info(f"Evidence: vs={evidence.vs}, sigma={evidence.sigma}")
            
            prior_variance = prior.variance()
            if prior_variance <= 0:
                logger.error(f"Prior variance invalid: {prior_variance}")
                raise UpdateError(f"Prior variance must be positive, got {prior_variance}")
            
            prior_precision = 1.0 / prior_variance
            measurement_precision = 1.0 / (evidence.sigma ** 2)
            
            posterior_precision = prior_precision + measurement_precision
            if posterior_precision <= 0:
                logger.error("Posterior precision is non-positive")
                raise UpdateError("Posterior precision is non-positive")
            
            posterior_mean = (
                prior_precision * prior.mean() + measurement_precision * evidence.vs
            ) / posterior_precision
            posterior_variance = 1.0 / posterior_precision
            
            logger.info(f"Update successful: posterior mean={posterior_mean:.4f}, variance={posterior_variance:.4f}")
            
            return GaussianBelief(posterior_mean, posterior_variance)
        except InvalidBeliefError:
            raise
        except Exception as e:
            logger.error(f"Update failed: {str(e)}", exc_info=True)
            raise UpdateError(f"Update failed: {str(e)}") from e

class BeliefStore:

    def __init__(self, lat_step: float = 0.1, lon_step: float = 0.1,
                 depth_step: float = 0.5, soilsat_step: float = 0.1) -> None:
        self._store: Dict[Tuple, GaussianBelief] = {}
        self._locks: Dict[Tuple, threading.Lock] = {}
        self._steps = (lat_step, lon_step, depth_step, soilsat_step)
        logger.info(f"BeliefStore initialized with grid steps {self._steps}")
    
    def _key(self, index: VsIndex) -> Tuple[int, int, int, int]:
        """Map a continuous index to a discrete grid cell."""
        lat_step, lon_step, depth_step, soilsat_step = self._steps
        return (
            round(index.lat / lat_step),
            round(index.lon / lon_step),
            round(index.depth / depth_step),
            round(index.soilsat / soilsat_step),
        )
    
    def get_belief(self, index: VsIndex) -> GaussianBelief | None:
        try:
            key = self._key(index)
            belief = self._store.get(key)
            if belief:
                logger.info(f"Retrieved belief at key {key}")
            else:
                logger.info(f"No belief found at key {key}")
            return belief
        except Exception as e:
            logger.error(f"Failed to retrieve belief: {str(e)}", exc_info=True)
            raise BeliefStoreError(f"Failed to retrieve belief: {str(e)}") from e
    
    def put_belief(self, index: VsIndex, belief: GaussianBelief) -> None:
        try:
            key = self._key(index)
            
            if key not in self._locks:
                self._locks[key] = threading.Lock()
            
            with self._locks[key]:
                self._store[key] = belief
                logger.info(f"Stored belief at key {key}")
        except Exception as e:
            logger.error(f"Failed to store belief: {str(e)}", exc_info=True)
            raise BeliefStoreError(f"Failed to store belief: {str(e)}") from e


def example_usage() -> None:
    store = BeliefStore()
    updater = PrecisionWeightedGaussianUpdate()
    
    try:
        logger.info("Starting example usage")
        index = VsIndex(lat=40.7, lon=-74.0, depth=-2.0, soilsat=0.3)
        prior = GaussianBelief(mean=2.5, variance=0.1)
        store.put_belief(index, prior)
        
        measurement = VsMeasurement(
            index=index,
            vs=2.7,
            sigma=0.15,
            timestamp=datetime.now(timezone.utc)
        )
        
        current_belief = store.get_belief(index)
        if current_belief is not None:
            updated_belief = updater.apply(current_belief, measurement)
            store.put_belief(index, updated_belief)
            logger.info(f"Updated mean: {updated_belief.mean():.4f}")
            logger.info(f"Updated variance: {updated_belief.variance():.4f}")
            logger.info("Example usage completed successfully")
        else:
            logger.warning("No current belief found for update")
    except (InvalidBeliefError, UpdateError, BeliefStoreError) as e:
        logger.error(f"Operation error: {e}")
    except ValidationError as e:
        logger.error(f"Validation error: {e}", exc_info=True)
    except Exception as e:
        logger.error(f"Unexpected error: {e}", exc_info=True)


if __name__ == "__main__":
    example_usage()


2026-05-04 22:45:30,666 - __main__ - INFO - BeliefStore initialized with grid steps (0.1, 0.1, 0.5, 0.1)
2026-05-04 22:45:30,667 - __main__ - INFO - Starting example usage
2026-05-04 22:45:30,668 - __main__ - INFO - GaussianBelief initialized with mean=2.5, variance=0.1
2026-05-04 22:45:30,668 - __main__ - INFO - Stored belief at key (407, -740, -4, 3)
2026-05-04 22:45:30,669 - __main__ - INFO - Retrieved belief at key (407, -740, -4, 3)
2026-05-04 22:45:30,669 - __main__ - INFO - Applying update at index lat=40.7 lon=-74.0 depth=-2.0 soilsat=0.3
2026-05-04 22:45:30,670 - __main__ - INFO - Prior: mean=2.5, variance=0.1
2026-05-04 22:45:30,670 - __main__ - INFO - Evidence: vs=2.7, sigma=0.15
2026-05-04 22:45:30,671 - __main__ - INFO - Update successful: posterior mean=2.6633, variance=0.0184
2026-05-04 22:45:30,671 - __main__ - INFO - GaussianBelief initialized with mean=2.663265306122449, variance=0.018367346938775512
2026-05-04 22:45:30,672 - __main__ - INFO - Stored belief at key (40

In [9]:
# Test error handling with logging: invalid variance
logger.info("Test 1: Invalid variance in GaussianBelief")
try:
    invalid_belief = GaussianBelief(mean=2.5, variance=-0.1)
except InvalidBeliefError as e:
    logger.error(f"Test 1 - Caught expected error: {e}")

# Test error handling with logging: very small variance (warning case)
logger.info("Test 2: Very small variance (warning case)")
try:
    small_var_belief = GaussianBelief(mean=2.5, variance=0.005)
    logger.info(f"Test 2 - Created belief with small variance: {small_var_belief.variance()}")
except InvalidBeliefError as e:
    logger.error(f"Test 2 - Caught unexpected error: {e}")

# Test error handling with logging: near-zero variance in update
logger.info("Test 3: Near-zero variance during update")
try:
    zero_var_belief = GaussianBelief(mean=2.5, variance=1e-10)
    index = VsIndex(lat=40.7, lon=-74.0, depth=-2.0, soilsat=0.3)
    measurement = VsMeasurement(
        index=index,
        vs=2.7,
        sigma=0.15,
        timestamp=datetime.now(timezone.utc)
    )
    updater = PrecisionWeightedGaussianUpdate()
    updater.apply(zero_var_belief, measurement)
    logger.info("Test 3 - Update completed")
except UpdateError as e:
    logger.error(f"Test 3 - Caught expected error: {e}")

# Test error handling with logging: successful store operations
logger.info("Test 4: BeliefStore with valid operations")
try:
    store = BeliefStore()
    index = VsIndex(lat=40.7, lon=-74.0, depth=-2.0, soilsat=0.3)
    belief = GaussianBelief(mean=2.5, variance=0.1)
    store.put_belief(index, belief)
    retrieved = store.get_belief(index)
    if retrieved:
        logger.info(f"Test 4 - Retrieved belief - mean: {retrieved.mean()}, variance: {retrieved.variance()}")
    else:
        logger.warning("Test 4 - No belief retrieved")
except BeliefStoreError as e:
    logger.error(f"Test 4 - Caught unexpected error: {e}")

logger.info("All tests completed")


2026-05-04 21:48:09,312 - __main__ - INFO - Test 1: Invalid variance in GaussianBelief
2026-05-04 21:48:09,314 - __main__ - ERROR - Invalid variance: -0.1
2026-05-04 21:48:09,314 - __main__ - ERROR - Test 1 - Caught expected error: Variance must be positive, got -0.1
2026-05-04 21:48:09,315 - __main__ - INFO - Test 2: Very small variance (warning case)
2026-05-04 21:48:09,315 - __main__ - WARNING - Variance is very small: 0.005
2026-05-04 21:48:09,316 - __main__ - INFO - GaussianBelief initialized with mean=2.5, variance=0.005
2026-05-04 21:48:09,316 - __main__ - INFO - Test 2 - Created belief with small variance: 0.005
2026-05-04 21:48:09,317 - __main__ - INFO - Test 3: Near-zero variance during update
2026-05-04 21:48:09,317 - __main__ - WARNING - Variance is very small: 1e-10
2026-05-04 21:48:09,318 - __main__ - INFO - GaussianBelief initialized with mean=2.5, variance=1e-10
2026-05-04 21:48:09,319 - __main__ - INFO - Applying update at index lat=40.7 lon=-74.0 depth=-2.0 soilsat=0.